In [1]:
import openmm as mm
from openmm import app, unit
from openmm.app.gromacsgrofile import GromacsGroFile

import numpy as np
import sys

from openmmnapshift.utils import RESIDUE_TYPES, get_napshift_force

This tutorial sets up a simulation of a short helical peptide with Chemical Shift restraints applied on top of the CHARMM27 forcefield. 

Staring from an extended conformation, the CS restraints should guide the system to a helical state.

## Define simulation parameters

In [2]:
temperature = 298*unit.kelvin
timestep = 2*unit.femtosecond
pressure = 1*unit.bar
collision_frequency = 1/unit.picosecond

max_K = 25
K_gradient = 0.001

report_interval = 1000

## Set up CHARMM27 system

In [3]:
gro = app.GromacsGroFile(f'From_Emma/V1_for_mina/V1/ions.gro')
top = app.GromacsTopFile(f'From_Emma/V1_for_mina/V1/topol.top', periodicBoxVectors=gro.getPeriodicBoxVectors(),
        includeDir=f'../Data/GromacsForceFields/top')
system = top.createSystem(nonbondedMethod=app.PME, nonbondedCutoff=1*unit.nanometer,
        constraints=app.HBonds)

system.addForce(mm.AndersenThermostat(temperature, collision_frequency))
system.addForce(mm.MonteCarloBarostat(pressure, temperature))

8

## Add Chemical Shift restraints

In [4]:
napshift_force = get_napshift_force(top.topology, 'From_Emma/V1_for_mina/V1/CS.txt', model_type='all_atom')
napshift_force.setUsesPeriodicBoundaryConditions(True)
system.addForce(napshift_force)

/home/mcul245/anaconda3/envs/BuildOpenMMNapShift/lib/python3.11/site-packages/pycamcoil/camcoil_engine.py:108: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  self.df[f_name] = read_csv(f_path, header=None, delim_whitespace=" ",
/home/mcul245/anaconda3/envs/BuildOpenMMNapShift/lib/python3.11/site-packages/pycamcoil/camcoil_engine.py:108: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  self.df[f_name] = read_csv(f_path, header=None, delim_whitespace=" ",
/home/mcul245/anaconda3/envs/BuildOpenMMNapShift/lib/python3.11/site-packages/pycamcoil/camcoil_engine.py:108: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  self.df[f_name] = read_csv(f_path, header=None, delim_whitespace=" ",
/home/mcul245/anaconda3/e

9

## Create the OpenMM simulation

In [5]:
integrator = mm.VerletIntegrator(timestep)
platform = mm.Platform.getPlatformByName("CUDA")
simulation = app.Simulation(top.topology, system, integrator, platform, {"Precision" : "mixed", 'DeviceIndex' : "0"})

TESTING TRICLINIC BOX


In [6]:
simulation.context.setPositions(gro.positions)
simulation.minimizeEnergy()
simulation.context.setVelocitiesToTemperature(temperature)

## Add reporters

In [7]:
xtc_reporter = app.XTCReporter('From_Emma/V1_for_mina/V1/output.xtc', report_interval, append=False, enforcePeriodicBox=True, atomSubset=[atom.index for atom in top.topology.atoms() if atom.residue.name not in ["HOH", "NA", "CL"]])
state_data_reporter = app.StateDataReporter(sys.stdout, report_interval, step=True, time=True, potentialEnergy=True, kineticEnergy=True, totalEnergy=True, temperature=True, volume=True, speed=True)
simulation.reporters.append(xtc_reporter)
simulation.reporters.append(state_data_reporter)

## Simulate!

In [8]:
warmup_steps = int(np.floor(max_K/K_gradient))
print(f"Warming up CS restraints for {len(range(warmup_steps))} steps")
for i in range(warmup_steps):
    simulation.step(1)
    simulation.context.setParameter('NapShift_K', (i*K_gradient))
    
print(f"Simulating with CS restraints")
simulation.step(100000)

Warming up CS restraints for 25000 steps
#"Step","Time (ps)","Potential Energy (kJ/mole)","Kinetic Energy (kJ/mole)","Total Energy (kJ/mole)","Temperature (K)","Box Volume (nm^3)","Speed (ns/day)"
1000,2.0000000000000013,-357709.7486119317,41959.53747543649,-315750.2111364952,216.45638502538193,232.23934356386258,0
2000,3.999999999999781,-347259.0412812049,47569.38634580502,-299689.6549353999,245.39587483099643,231.81959787416133,374
3000,5.999999999999561,-341019.92856393266,51042.78943589605,-289977.1391280366,263.3140960949279,230.6065114109891,381
4000,7.999999999999341,-335242.8255478118,53267.15718617015,-281975.66836164164,274.7889271145318,231.09030540921202,382
5000,10.000000000000009,-331513.0892438912,55319.6848234324,-276193.40442045877,285.37728769373416,232.95894204926375,383
6000,12.000000000000677,-328487.4169599642,55914.59754756018,-272572.81941240403,288.4462599803254,233.3869797299516,385
7000,14.000000000001345,-326470.0188674878,56573.061247568476,-269896.95761991

The output trajectory will be written to Data/ShortHelix

In [6]:
from openmmnapshift.utils import parse_BMRB_entry, read_chemical_shifts

In [7]:
parse_BMRB_entry(27473, 'snowflea')

exp_CS = read_chemical_shifts('snowflea/27473_CS_0.txt')

In [8]:
exp_CS

{('2', '.'): ('K',
  {'CA': 56.19, 'CB': 32.74, 'C': nan, 'H': nan, 'HA': 4.03, 'N': 119.3},
  {'CA': 1.0, 'CB': 1.0, 'C': 1.0, 'H': 1.0, 'HA': 1.0, 'N': 1.0}),
 ('3', '.'): ('G',
  {'CA': 44.68, 'CB': nan, 'C': nan, 'H': nan, 'HA': nan, 'N': 110.7},
  {'CA': 1.0, 'CB': 1.0, 'C': 1.0, 'H': 1.0, 'HA': 1.0, 'N': 1.0}),
 ('4', '.'): ('A',
  {'CA': 52.03, 'CB': 19.1, 'C': nan, 'H': nan, 'HA': 4.276, 'N': 126.2},
  {'CA': 1.0, 'CB': 1.0, 'C': 1.0, 'H': 1.0, 'HA': 1.0, 'N': 1.0}),
 ('5', '.'): ('D',
  {'CA': 53.67, 'CB': 41.04, 'C': nan, 'H': nan, 'HA': 4.74, 'N': 119.8},
  {'CA': 1.0, 'CB': 1.0, 'C': 1.0, 'H': 1.0, 'HA': 1.0, 'N': 1.0}),
 ('6', '.'): ('G',
  {'CA': 44.59, 'CB': nan, 'C': nan, 'H': nan, 'HA': nan, 'N': 108.2},
  {'CA': 1.0, 'CB': 1.0, 'C': 1.0, 'H': 1.0, 'HA': 1.0, 'N': 1.0}),
 ('7', '.'): ('A',
  {'CA': 52.22, 'CB': 19.1, 'C': nan, 'H': nan, 'HA': 4.014, 'N': 125.7},
  {'CA': 1.0, 'CB': 1.0, 'C': 1.0, 'H': 1.0, 'HA': 1.0, 'N': 1.0}),
 ('8', '.'): ('H',
  {'CA': 55.6, 'CB': 